# Notebook 05 — Feature Engineering
**Project:** Loan Default Risk Analysis and Prediction  
**Phase:** 6a of 8  
**Objective:** Build the scikit-learn preprocessing pipeline, inspect encoded features, check for data leakage, and confirm the feature matrix is ready for model training.

---

## 0. Environment Setup

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

from src.data_loader import load_data
from src.data_cleaner import clean_data
from src.feature_engineering import build_preprocessor, get_feature_names
from src.config import (
    RAW_DATA_PATH, TARGET_COLUMN, RANDOM_STATE, TEST_SIZE,
    NUMERIC_FEATURES, CATEGORICAL_FEATURES, BINARY_FEATURES,
    FIGURES_DIR, TABLES_DIR
)

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
print('Environment ready.')

---
## 1. Load Clean Data

In [ ]:
df = clean_data(load_data(RAW_DATA_PATH))
print(f'Shape: {df.shape}')
df.head(3)

---
## 2. Build Feature Matrix (X) and Target (y)

In [ ]:
X, y, preprocessor = build_preprocessor(df)
print(f'X shape : {X.shape}')
print(f'y shape : {y.shape}')
print(f'y balance: {y.value_counts(normalize=True).round(3).to_dict()}')
print(f'Feature columns: {X.columns.tolist()}')

---
## 3. Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
print(f'Train set: {X_train.shape[0]:,} rows')
print(f'Test set : {X_test.shape[0]:,} rows')
print(f'Train default rate: {y_train.mean()*100:.2f}%')
print(f'Test  default rate: {y_test.mean()*100:.2f}%')

---
## 4. Fit Preprocessor on Training Data Only

In [ ]:
# IMPORTANT: fit ONLY on training data to prevent data leakage
preprocessor.fit(X_train)
feature_names = get_feature_names(preprocessor)
print(f'Output feature count: {len(feature_names)}')
print(f'Feature names: {feature_names}')

---
## 5. Inspect Transformed Training Data

In [ ]:
X_train_transformed = preprocessor.transform(X_train)
print(f'Transformed shape: {X_train_transformed.shape}')
print()
print('Summary statistics after transformation:')
X_train_transformed.describe().round(3)

---
## 6. Numeric Feature Scaling Verification

In [ ]:
# Verify scaled numeric features have mean ~0 and std ~1
numeric_transformed = X_train_transformed[NUMERIC_FEATURES]
print('After StandardScaler — means should be ~0, stds should be ~1:')
check = pd.DataFrame({
    'Mean': numeric_transformed.mean().round(4),
    'Std':  numeric_transformed.std().round(4),
    'Min':  numeric_transformed.min().round(4),
    'Max':  numeric_transformed.max().round(4),
})
check

---
## 7. Categorical Encoding Verification

In [ ]:
cat_transformed = X_train_transformed[CATEGORICAL_FEATURES]
print('Categorical features — encoded unique values:')
for col in CATEGORICAL_FEATURES:
    uniq = sorted(cat_transformed[col].unique())
    print(f'  {col}: {uniq}')

---
## 8. Class Imbalance Visualisation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (split_name, y_split) in zip(axes, [('Train', y_train), ('Test', y_test)]):
    counts = y_split.value_counts().sort_index()
    ax.bar(['No Default', 'Default'], counts.values,
           color=['#4a90d9', '#e05c5c'], edgecolor='white')
    for i, v in enumerate(counts.values):
        ax.text(i, v + 100, f'{v:,}', ha='center', fontsize=9)
    ax.set_title(f'{split_name} Set Class Distribution', fontweight='bold')
    ax.set_ylabel('Count')

plt.suptitle('Class Balance in Train / Test Splits', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fe_class_balance.png', bbox_inches='tight')
plt.show()
print('Saved -> outputs/figures/fe_class_balance.png')

---
## 9. Feature Engineering Summary

| Step | Detail |
|---|---|
| Numeric features | StandardScaler (zero mean, unit variance) |
| Categorical features | OrdinalEncoder |
| Binary features | Passthrough (already 0/1) |
| Total features | 16 |
| Train rows | *(fill after run)* |
| Test rows | *(fill after run)* |
| Data leakage check | Preprocessor fitted on train only |

---
**Next:** Notebook 06 — Model Training & Evaluation